# Polars → HF Buckets: Streaming Parquet Sink Demo

This notebook demonstrates a proof-of-concept streaming parquet sink from Polars to HuggingFace Buckets via the XET protocol.

**What it does**: `sink_parquet("hf://buckets/...")` streams parquet data directly to an HF Bucket with constant memory usage — O(row_group_size), not O(dataset_size).

**Prerequisites**:
- A HuggingFace account with a [Bucket](https://huggingface.co/new-bucket) created
- An HF token with write access (add it to Colab Secrets as `HF_TOKEN`)
- The custom Polars wheels (built from the `feature/hf-bucket-sink` branch)

## 1. Install Custom Wheels

Install the custom Polars wheels built from the `feature/hf-bucket-sink` branch.

**Important**: Use `--no-deps --force-reinstall` to prevent pip from replacing the custom `polars-runtime-32` wheel with the upstream PyPI version (they share the same version number).

After installing, **restart the runtime** (Runtime → Restart runtime) before continuing.

In [ ]:
HF_WHEEL_REPO = "https://huggingface.co/datasets/davanstrien/polars-hf-bucket-sink-wheels/resolve/main"

!pip uninstall polars polars-runtime-32 -y -q
!pip install --no-deps --force-reinstall \
    "{HF_WHEEL_REPO}/polars-1.38.1-py3-none-any.whl" \
    "{HF_WHEEL_REPO}/polars_runtime_32-1.38.1-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl" \
    -q

**⚠️ Restart the runtime now** (Runtime → Restart runtime), then continue from cell 2.

## 2. Setup

In [ ]:
import polars as pl
import os

# Load HF_TOKEN from Colab Secrets
from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

print(f"Polars version: {pl.__version__}")
print(f"HF token loaded: {'HF_TOKEN' in os.environ}")

In [ ]:
# ── Configure your bucket here ──
# Create a bucket at https://huggingface.co/new-bucket

NAMESPACE = "<YOUR_NAMESPACE>"  # your HF username or org
BUCKET = "<YOUR_BUCKET>"        # bucket name

BUCKET_BASE = f"hf://buckets/{NAMESPACE}/{BUCKET}"

## 3. Example 1: Simple Write

The simplest case — create a DataFrame and sink it directly to a bucket.

In [ ]:
df = pl.DataFrame({
    "id": [1, 2, 3],
    "name": ["alice", "bob", "charlie"],
    "score": [0.95, 0.87, 0.92],
})

df.lazy().sink_parquet(
    f"{BUCKET_BASE}/simple-test.parquet",
    storage_options={"token": os.environ["HF_TOKEN"]},
)

print("Written to bucket.")

In [ ]:
# Read back and verify the data round-trips correctly
from huggingface_hub import download_bucket_files
import tempfile

with tempfile.TemporaryDirectory() as tmpdir:
    download_bucket_files(
        f"{NAMESPACE}/{BUCKET}",
        files=[("simple-test.parquet", f"{tmpdir}/simple-test.parquet")],
        token=os.environ["HF_TOKEN"],
    )
    result = pl.read_parquet(f"{tmpdir}/simple-test.parquet")

print(result)

## 4. Example 2: Hub Is Your Disk

The key pattern — scan from a public HF dataset, transform, and sink to a bucket.

This runs fully streaming: data flows from the Hub → through Polars → to the bucket without buffering the full dataset in memory.

In [ ]:
storage_options = {"token": os.environ["HF_TOKEN"]}

(
    pl.scan_parquet(
        "hf://datasets/wikimedia/wikipedia/20231101.en/train-00000-of-00041.parquet",
        storage_options=storage_options,
    )
    .filter(pl.col("text").str.len_chars() > 5000)
    .select("id", "url", "title", "text")
    .sink_parquet(
        f"{BUCKET_BASE}/wikipedia-long-articles.parquet",
        storage_options=storage_options,
    )
)

print("Scan -> filter -> sink complete.")

## 5. Example 3: FineWeb-Edu ETL at Scale

Filter the FineWeb-Edu 10BT sample (88 parquet shards) — keep only high-education, short documents — and sink directly to a bucket.

This is the "hub is your disk" pattern at real scale. On Colab this takes ~22 min and produces a ~310 MB parquet file.

## Notes

- **This is a PoC** — single-file output, requires custom wheels built from the feature branch.
- **Token refresh**: XET upload tokens are automatically refreshed for long-running uploads.
- **Memory model**: O(row_group_size), not O(dataset_size). Parquet bytes stream through a bounded channel to the XET upload.
- **Branch**: [`feature/hf-bucket-sink`](https://github.com/davanstrien/polars/tree/feature/hf-bucket-sink)
- **Core diff**: ~52 lines in 7 polars-stream/polars-io files, all `#[cfg(feature = "hf_bucket_sink")]` gated.
- **E2E tests**: `py-polars/tests/unit/io/cloud/test_hf_bucket_sink.py` — smoke, 10K, and 10M row tests (gated behind `HF_TOKEN` + `pytest.mark.slow`).
- **Read support**: `pl.read_parquet("hf://buckets/...")` is not yet supported (separate concern).